# Notebook Belajar: Recommendation Scoring (Supervised Regression)

Notebook ini dibuat untuk pemula agar Anda bisa memahami alur end-to-end machine learning pada kasus **rekomendasi produk e-commerce**.

Target yang diprediksi:
- `Probability for the product to be recommended to the person` (nilai 0 sampai 1)

Di notebook ini Anda akan belajar:
1. Membaca dan memahami struktur dataset
2. Membersihkan nama kolom agar konsisten (`snake_case`)
3. Melakukan validasi data + exploratory checks
4. Melakukan preprocessing numerik dan kategorikal
5. Melihat **statistik saat preprocessing** (imputasi, scaling, one-hot encoding)
6. Melatih beberapa model baseline
7. Membandingkan performa model (MAE, RMSE, R2)
8. Menyimpan model terbaik dan metadata
        


## 0) Persiapan

Pastikan Anda sudah menjalankan:
```bash
pip install -r ../requirements.txt
```

Jika notebook dijalankan dari folder `project/notebooks`, path dataset akan diarahkan otomatis ke `../data/content_based_recommendation_dataset.csv`.
        


In [ ]:
from __future__ import annotations

import json
import re
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

RANDOM_STATE = 42
TEST_SIZE = 0.2
RECOMMENDATION_THRESHOLD = 0.7

print("Import berhasil.")
        


## 1) Load Dataset

Kita baca dataset CSV dari folder `data` dan cek ukuran awal data.
        


In [ ]:
BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    PROJECT_DIR = BASE_DIR.parent
else:
    PROJECT_DIR = BASE_DIR

DATA_PATH = PROJECT_DIR / "data" / "content_based_recommendation_dataset.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset tidak ditemukan di: {DATA_PATH}")

raw_df = pd.read_csv(DATA_PATH)
print(f"Dataset path : {DATA_PATH}")
print(f"Shape        : {raw_df.shape}")
raw_df.head()
        


## 2) Bersihkan Nama Kolom (Original -> Snake Case)

Mengapa penting?
- Nama kolom asli panjang dan memiliki spasi/simbol.
- Untuk coding API/pipeline, nama `snake_case` lebih konsisten dan aman.
        


In [ ]:
def to_snake_case(column_name: str) -> str:
    normalized = re.sub(r"[^0-9a-zA-Z]+", "_", column_name).strip("_").lower()
    return re.sub(r"_+", "_", normalized)

column_mapping = {col: to_snake_case(col) for col in raw_df.columns}

mapping_df = pd.DataFrame(
    {
        "original_column": list(column_mapping.keys()),
        "cleaned_column": list(column_mapping.values()),
    }
)

clean_df = raw_df.rename(columns=column_mapping).copy()
mapping_df
        


## 3) Validasi Dasar Data + Exploratory Checks Singkat

Di tahap ini kita lihat:
- tipe data
- missing value
- duplikasi
- ringkasan statistik target
        


In [ ]:
target_original = "Probability for the product to be recommended to the person"
target_col = to_snake_case(target_original)

if target_col not in clean_df.columns:
    raise ValueError(f"Target column tidak ditemukan: {target_col}")

rows, cols = clean_df.shape
missing_total = int(clean_df.isna().sum().sum())
duplicated_rows = int(clean_df.duplicated().sum())

summary = {
    "rows": rows,
    "columns": cols,
    "missing_values_total": missing_total,
    "duplicated_rows": duplicated_rows,
    "target_min": float(clean_df[target_col].min()),
    "target_max": float(clean_df[target_col].max()),
    "target_mean": float(clean_df[target_col].mean()),
    "target_std": float(clean_df[target_col].std()),
}

print("Ringkasan dataset:")
for k, v in summary.items():
    print(f"- {k}: {v}")

print()
print("Tipe data tiap kolom:")
print(clean_df.dtypes)

print()
print("Missing value per kolom:")
print(clean_df.isna().sum())
        


In [ ]:
print("Statistik deskriptif numerik:")
clean_df.describe().T
        


## 4) Definisikan Fitur dan Split Train/Test

Pemisahan data train/test penting agar evaluasi mencerminkan performa pada data baru.
        


In [ ]:
feature_cols = [c for c in clean_df.columns if c != target_col]

numeric_features = clean_df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_features = clean_df[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()

X = clean_df[feature_cols].copy()
y = clean_df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

print(f"Jumlah fitur total       : {len(feature_cols)}")
print(f"- Numerik               : {len(numeric_features)} -> {numeric_features}")
print(f"- Kategorikal           : {len(categorical_features)} -> {categorical_features}")
print(f"Train shape             : {X_train.shape}")
print(f"Test shape              : {X_test.shape}")
        


## 5) Bangun Preprocessing Pipeline

Strategi yang dipakai:
- Numerik: `SimpleImputer(strategy="median")` + `StandardScaler()`
- Kategorikal: `SimpleImputer(strategy="most_frequent")` + `OneHotEncoder(handle_unknown="ignore")`

Kenapa `handle_unknown="ignore"` penting?
- Jika ada kategori baru saat prediksi API (misalnya brand baru), model tidak crash.
        


In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)

print("Preprocessor siap.")
        


## 6) Statistik Saat Preprocessing

Pada bagian ini kita lihat statistik penting selama preprocessing:
1. Missing value sebelum preprocessing
2. Nilai median yang dipakai imputer numerik
3. Nilai paling sering (mode) untuk imputer kategorikal
4. Mean/std fitur numerik sebelum dan sesudah scaling
5. Jumlah fitur hasil one-hot encoding
        


In [ ]:
# Fit preprocessor hanya pada data train (best practice)
preprocessor.fit(X_train)

# Transform train/test
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Missing sebelum preprocessing
missing_before = X_train.isna().sum().rename("missing_count_train").to_frame()

# Akses komponen preprocessing yang sudah fit
num_imputer = preprocessor.named_transformers_["numeric"].named_steps["imputer"]
cat_imputer = preprocessor.named_transformers_["categorical"].named_steps["imputer"]
cat_encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]

# Statistik imputasi numerik (median)
num_median_df = pd.DataFrame(
    {
        "feature": numeric_features,
        "median_used_for_imputation": num_imputer.statistics_.astype(float),
        "mean_before_scaling": X_train[numeric_features].mean().values,
        "std_before_scaling": X_train[numeric_features].std(ddof=0).values,
    }
)

# Setelah scaling (hanya bagian numerik)
num_after_scaling = preprocessor.named_transformers_["numeric"].transform(X_train[numeric_features])
num_median_df["mean_after_scaling"] = num_after_scaling.mean(axis=0)
num_median_df["std_after_scaling"] = num_after_scaling.std(axis=0)

# Statistik imputasi kategorikal (most frequent)
cat_mode_df = pd.DataFrame(
    {
        "feature": categorical_features,
        "most_frequent_used_for_imputation": cat_imputer.statistics_.astype(str),
        "n_unique_train_before_encoding": [X_train[c].nunique(dropna=True) for c in categorical_features],
    }
)

# Statistik one-hot
ohe_feature_count = sum(len(cats) for cats in cat_encoder.categories_)
stats_preprocessing = {
    "train_rows": X_train.shape[0],
    "test_rows": X_test.shape[0],
    "numeric_input_features": len(numeric_features),
    "categorical_input_features": len(categorical_features),
    "one_hot_generated_features": int(ohe_feature_count),
    "total_transformed_features": int(X_train_processed.shape[1]),
}

print("Statistik preprocessing ringkas:")
for k, v in stats_preprocessing.items():
    print(f"- {k}: {v}")

print()
print("Missing value sebelum preprocessing (train):")
display(missing_before)

print()
print("Statistik numerik (imputasi + scaling):")
display(num_median_df)

print()
print("Statistik kategorikal (imputasi + encoding):")
display(cat_mode_df)
        


In [ ]:
# Nama fitur hasil transformasi (berguna untuk debugging/interpretasi)
feature_names_out = preprocessor.get_feature_names_out()
print(f"Jumlah feature names output: {len(feature_names_out)}")
print("Contoh 30 fitur pertama:")
print(feature_names_out[:30])
        


## 7) Training Baseline Models + Evaluasi

Model baseline yang dibandingkan:
- LinearRegression
- RandomForestRegressor
- GradientBoostingRegressor

Metrik evaluasi:
- MAE (lebih kecil lebih baik)
- RMSE (lebih kecil lebih baik)
- R2 (lebih besar lebih baik)
        


In [ ]:
def regression_metrics(y_true, y_pred):
    return {
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }

candidate_models = {
    "LinearRegression": LinearRegression(),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

# Tambahan opsional: XGBoost (jika dependency tersedia)
try:
    from xgboost import XGBRegressor

    candidate_models["XGBRegressor"] = XGBRegressor(
        n_estimators=400,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
    )
    print("XGBRegressor tersedia dan ikut dievaluasi.")
except Exception:
    print("XGBRegressor tidak tersedia. Lanjut dengan baseline sklearn.")

result_rows = []
trained_pipelines = {}

for model_name, model in candidate_models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    metrics = regression_metrics(y_test, y_pred)
    result_rows.append({"model": model_name, **metrics})
    trained_pipelines[model_name] = pipeline

results_df = pd.DataFrame(result_rows).sort_values(by="rmse", ascending=True).reset_index(drop=True)
results_df
        


## 8) Pilih Model Terbaik

Kriteria pemilihan: **RMSE terendah**.
        


In [ ]:
best_model_name = results_df.iloc[0]["model"]
best_pipeline = trained_pipelines[best_model_name]

print(f"Model terbaik (berdasarkan RMSE): {best_model_name}")
print(results_df.iloc[0])
        


## 9) Contoh Prediksi 1 Record

Kita coba prediksi satu data contoh, lalu konversi ke label rekomendasi:
- `recommended` jika probabilitas >= 0.7
- `not_recommended` jika < 0.7
        


In [ ]:
sample_payload = {
    "number_of_clicks_on_similar_products": 12,
    "number_of_similar_products_purchased_so_far": 4,
    "average_rating_given_to_similar_products": 4.2,
    "gender": "male",
    "median_purchasing_price_in_rupees": 500,
    "rating_of_the_product": 4.5,
    "brand_of_the_product": "PUMA",
    "customer_review_sentiment_score_overall": 0.8,
    "price_of_the_product": 200,
    "holiday": "No",
    "season": "winter",
    "geographical_locations": "plains",
}

sample_df = pd.DataFrame([sample_payload])[feature_cols]
pred = float(best_pipeline.predict(sample_df)[0])
pred_clipped = float(np.clip(pred, 0.0, 1.0))
label = "recommended" if pred_clipped >= RECOMMENDATION_THRESHOLD else "not_recommended"

print(f"Predicted probability : {pred_clipped:.6f}")
print(f"Recommendation label  : {label}")
        


## 10) Simpan Model + Metadata

Simpan artefak ke folder `models/` agar bisa dipakai oleh Flask API.
        


In [ ]:
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / "best_model.joblib"
METADATA_PATH = MODELS_DIR / "model_metadata.json"

joblib.dump(best_pipeline, MODEL_PATH)

metadata = {
    "selected_model": best_model_name,
    "selection_metric": "rmse",
    "metrics": results_df.set_index("model").to_dict(orient="index"),
    "feature_columns": feature_cols,
    "numeric_feature_columns": numeric_features,
    "categorical_feature_columns": categorical_features,
    "target_column": target_col,
    "column_mapping": column_mapping,
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "training_timestamp": datetime.now(timezone.utc).isoformat(),
    "preprocessing_statistics": stats_preprocessing,
}

with METADATA_PATH.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print(f"Model tersimpan    : {MODEL_PATH}")
print(f"Metadata tersimpan : {METADATA_PATH}")
        


## 11) Interpretasi Hasil (Untuk Pemula)

Panduan membaca metrik:
- **MAE**: rata-rata selisih absolut prediksi vs nilai asli. Semakin kecil semakin baik.
- **RMSE**: mirip MAE, tapi lebih sensitif ke error besar. Semakin kecil semakin baik.
- **R2**: proporsi variasi target yang bisa dijelaskan model. Semakin mendekati 1 semakin baik.

Jika model tree-based (misalnya RandomForest) menang:
- Biasanya karena hubungan fitur-target bersifat non-linear.
- Interaksi antar fitur bisa ditangkap lebih baik daripada model linear sederhana.

Next step agar lebih kuat:
1. Tambahkan cross-validation
2. Hyperparameter tuning
3. Monitoring drift data saat deployment
4. Buat evaluasi per segmen (misal per season/region/brand)
        
